# 第 5 章　统计聚合与线性代数

> 📖 **本章目标**
>
> - 彻底理解 axis 的聚合语义：聚合就是“沿某个轴压缩、消灭该维度”
> - 掌握聚合函数全家福：sum / mean / std / min / argmin / cumsum / percentile ……
> - 学会用 keepdims 保住维度，为后续广播铺路
> - 掌握 NaN 处理：检测、nan-safe 函数家族、填充策略
> - 补全数学函数：三角函数、指数对数、hypot
> - 攻克林代数：`A @ B` vs `A * B`、det / inv / eig / solve / lstsq
> - 用最小二乘做一次真实的线性拟合，感受 NumPy 完整工作流

> 🔗 这是系列的第 5 章。前四章我们依次掌握了：ndarray 基础与 dtype（第 1 章）、索引切片与布尔筛选（第 2 章）、向量化运算与广播（第 3 章）、形状变换与拼接（第 4 章）。从本章开始，我们从“会操作数组”迈向“会用数组做统计与分析”。

In [1]:
import numpy as np
print(np.__version__)

2.4.4


## 5.1　axis 的聚合语义

在第 3 章我们学过：axis 就是“沿着哪个方向看数据”。本章的聚合（aggregation）函数——`sum`、`mean`、`std`……——几乎都要用到 axis，所以先把 axis 的**聚合语义**讲透。

### 心智模型：聚合 = 沿 axis 压缩、消灭该维度

想象一个 (3, 4) 的二维数组，它像一张 3 行 4 列的表。`sum(axis=0)` 的意思是“**沿着第 0 轴（行方向）一路加过去，把这一维压扁**”：

- 结果只剩 4 列 → 形状 (4,)，相当于“把每一列的数全部加起来，得到 4 个列和”
- 同理 `sum(axis=1)` 是“沿着第 1 轴（列方向）压缩”，结果只剩 3 行 → 形状 (3,)，得到 3 个行和

> 💡 **一句话记忆**：`axis=k` 的聚合，就是“把第 k 个维度**消灭**掉”。结果的维度数比原来少 1。`axis=None`（默认）则是把所有维度都消灭，聚合出一个标量。

下面这张 mermaid 图画出了 (3, 4) 数组沿不同 axis 聚合后的形状变化：

```mermaid
flowchart LR
    A["原数组 (3, 4)"] --> B["sum(axis=0)<br/>沿第0轴(行)压缩<br/>把每一列相加"]
    A --> C["sum(axis=1)<br/>沿第1轴(列)压缩<br/>把每一行相加"]
    A --> D["sum() / sum(axis=None)<br/>全部压缩<br/>所有元素相加"]
    B --> B1["结果 (4,)<br/>4个列和"]
    C --> C1["结果 (3,)<br/>3个行和"]
    D --> D1["标量<br/>全体元素之和"]
```

In [2]:
# 先造一个 (3, 4) 的二维数组
A = np.arange(12).reshape(3, 4)
print('A 的形状:', A.shape)
print(A)

print()
# 沿 axis=0：把“行”这一维消灭，得到 4 个列和
print('sum(axis=0) 形状:', A.sum(axis=0).shape, '=>', A.sum(axis=0))

print()
# 沿 axis=1：把“列”这一维消灭，得到 3 个行和
print('sum(axis=1) 形状:', A.sum(axis=1).shape, '=>', A.sum(axis=1))

print()
# axis=None：全部消灭，得到标量
print('sum() 全部相加 =>', A.sum())

A 的形状: (3, 4)
[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]

sum(axis=0) 形状: (4,) => [12 15 18 21]

sum(axis=1) 形状: (3,) => [ 6 22 38]

sum() 全部相加 => 66


## 5.2　聚合函数全家福

聚合函数数量很多，但用法高度一致：**都接受 axis、keepdims 等参数，都支持全数组聚合**。下面按“求和 / 均值 / 标准差 / 最值 / 累积 / 分位数”六组过一遍。

### ① 求和类：sum / prod

`sum` 求和、`prod` 求连乘，是聚合的“元祖”。prod 用得少，但在概率计算（连乘）中很关键。

In [3]:
# 求和 / 连乘
arr = np.array([1, 2, 3, 4])
print('sum:', arr.sum())        # 1+2+3+4
print('prod:', arr.prod())      # 1*2*3*4

# 二维数组沿 axis 求和的两种写法等价
A = np.arange(6).reshape(2, 3)
print()
print('A =')
print(A)
print('A.sum(axis=0):', A.sum(axis=0), ' == np.sum(A, axis=0):', np.sum(A, axis=0))

sum: 10
prod: 24

A =
[[0 1 2]
 [3 4 5]]
A.sum(axis=0): [3 5 7]  == np.sum(A, axis=0): [3 5 7]


### ② 均值类：mean / median / average

- `mean`：算术平均
- `median`：中位数（对异常值更鲁棒）
- `average`：支持**加权平均**（weights 参数），数据清洗里很有用

In [4]:
scores = np.array([88, 92, 55, 100, 61])
print('mean:', scores.mean())
print('median:', np.median(scores))     # 排序后取中间值

# 加权平均：平时分和考试分各占不同权重
usual, exam = 80, 90
weights = np.array([0.3, 0.7])
print('加权平均:', np.average([usual, exam], weights=weights))

mean: 79.2
median: 88.0
加权平均: 87.0


### ③ 标准差：std / var，ddof 自由度参数详解

方差 `var` 与标准差 `std = sqrt(var)` 衡量数据的**离散程度**。其中 `ddof`（delta degrees of freedom）是最容易搞混的参数：

| 参数 | 分母 | 含义 | 适用场景 |
|------|------|------|----------|
| `ddof=0`（默认） | 除以 N | **总体标准差**：数据就是全部总体 | 描述“这批数据本身”散布多大 |
| `ddof=1` | 除以 N-1 | **样本标准差**（无偏估计） | 用样本去**推断**总体 |

> 💡 **直觉**：样本估计总体时，样本均值通常略“贴合”样本本身，导致方差被低估，所以分母用 N-1 稍微放大一点，做无偏修正。数据量越大，两者差别越小。

> ⚠️ **陷阱**：`np.std()` 默认是**总体标准差**，而 Excel 的 `STDEV.S`、pandas 的 `std()` 默认是**样本标准差**！跨工具对比数值时务必先确认口径。

In [5]:
data = np.array([2.0, 4.0, 6.0, 8.0, 10.0])
N = len(data)

# ddof=0：总体标准差（默认）
print('std(ddof=0) 总体:', np.std(data))
print('  手算 sqrt(sum((x-mean)^2)/N):', np.sqrt(((data - data.mean())**2).sum() / N))

# ddof=1：样本标准差
print('std(ddof=1) 样本:', np.std(data, ddof=1))
print('var(ddof=0):', np.var(data), ' var(ddof=1):', np.var(data, ddof=1))

std(ddof=0) 总体: 2.8284271247461903
  手算 sqrt(sum((x-mean)^2)/N): 2.8284271247461903
std(ddof=1) 样本: 3.1622776601683795
var(ddof=0): 8.0  var(ddof=1): 10.0


### ④ 最值与索引：min / max / argmin / argmax / ptp

- `min` / `max`：最小值 / 最大值
- `argmin` / `argmax`：**最值出现的位置（索引）**，注意默认返回“扁平化后的下标”
- `ptp`（peak-to-peak）：`max - min`，即极差，描述取值范围

In [6]:
arr = np.array([3, 1, 4, 1, 5, 9, 2, 6])
print('min:', arr.min(), ' max:', arr.max())
print('argmin 位置:', arr.argmin(), ' argmax 位置:', arr.argmax())
print('ptp 极差:', np.ptp(arr), ' == max-min:', arr.max() - arr.min())

# argmax 配合布尔筛选：找出第一个大于阈值的位置
thr = 5
idx = np.argmax(arr > thr)          # 第一个 True 的位置
print('第一个大于', thr, '的元素:', arr[idx], '(位置', idx, ')')

min: 1  max: 9
argmin 位置: 1  argmax 位置: 5
ptp 极差: 8  == max-min: 8
第一个大于 5 的元素: 9 (位置 5 )


### ⑤ 累积函数：cumsum / cumprod（不压缩维度）

前面所有聚合函数都是“压缩维度”。**累积函数恰恰相反：结果和原数组形状一样**，它把“截至目前为止的累积和”写进每个位置。

| 函数 | 输出形状 | 含义 |
|------|----------|------|
| `sum` | 标量 / 更少维 | 全体求和（压缩） |
| `cumsum` | 同输入 | 每个位置 = 到它为止的前缀和（不压缩） |
| `cumprod` | 同输入 | 每个位置 = 到它为止的前缀积（不压缩） |

> 💡 实战用途：`cumsum` 常用来做“累计占比”“分段累加”“复利增长曲线”等。

In [7]:
arr = np.array([1, 2, 3, 4])
print('cumsum:', np.cumsum(arr))     # [1, 1+2, 1+2+3, 1+2+3+4]
print('cumprod:', np.cumprod(arr))   # [1, 2, 6, 24]
print('sum:', arr.sum())             # 对比：压缩成标量

A = np.arange(6).reshape(2, 3)
print()
print('A =')
print(A)
print('cumsum(axis=1) 形状不变:', np.cumsum(A, axis=1))

cumsum: [ 1  3  6 10]
cumprod: [ 1  2  6 24]
sum: 10

A =
[[0 1 2]
 [3 4 5]]
cumsum(axis=1) 形状不变: [[ 0  1  3]
 [ 3  7 12]]


### ⑥ 分位数：percentile / quantile

`percentile` 用百分比（0~100），`quantile` 用 0~1 的小数，两者本质一样。分位数描述“数据排在哪个水位线”——`percentile(50)` 就是中位数。

In [8]:
data = np.sort(np.array([23, 45, 67, 12, 89, 34, 56, 78]))
print('排序后:', data)
print('p50 中位数:', np.percentile(data, 50))
print('p25 / p75:', np.percentile(data, [25, 75]))
print('quantile(0.5):', np.quantile(data, 0.5))

排序后: [12 23 34 45 56 67 78 89]
p50 中位数: 50.5
p25 / p75: [31.25 69.75]
quantile(0.5): 50.5


## 5.3　keepdims：聚合完别把维度弄丢

默认情况下聚合会“消灭”维度——比如 (3, 4) 的数组沿 axis=1 聚合后变成 (3,)。**但在做“按行归一化”这类操作时，我们恰恰需要 (3, 1) 而不是 (3,)**，因为广播（第 3 章）要求“对应维度要么相等、要么是 1”。

`keepdims=True` 就是让结果**保留一个长度为 1 的维度**，方便直接广播。

In [9]:
A = np.array([[1.0, 2.0, 3.0],
              [4.0, 5.0, 6.0]])

row_sum    = A.sum(axis=1)                 # 形状 (2,)：行和
row_sum_kd = A.sum(axis=1, keepdims=True)  # 形状 (2, 1)：保留了维度

print('普通 sum(axis=1):', row_sum, ' 形状', row_sum.shape)
print('keepdims sum(axis=1):', row_sum_kd, ' 形状', row_sum_kd.shape)

# 按行归一化：每行元素都除以该行的和，行和要 (2,1) 才能广播
row_norm = A / row_sum_kd
print()
print('按行归一化:')
print(row_norm)
print('每行之和应为1:', row_norm.sum(axis=1))

普通 sum(axis=1): [ 6. 15.]  形状 (2,)
keepdims sum(axis=1): [[ 6.]
 [15.]]  形状 (2, 1)

按行归一化:
[[0.16666667 0.33333333 0.5       ]
 [0.26666667 0.33333333 0.4       ]]
每行之和应为1: [1. 1.]


## 5.4　NaN 处理：数据清洗的必修课

真实数据几乎一定含缺失值，NumPy 用 `NaN`（Not a Number，非数值）表示。**NaN 最大的特点是“传染性”**：任何运算，只要碰到 NaN，结果就是 NaN。

> ⚠️ **陷阱**：`NaN == NaN` 是 `False`！所以不能用 `==` 判断缺失，必须用 `np.isnan()`。

In [10]:
x = np.array([1.0, np.nan, 3.0])
print('x:', x)

# NaN 的传染性
print('x + 10:', x + 10)                 # 含 NaN 的位置结果也是 NaN
print('x.sum():', x.sum())               # 一个 NaN 毁掉整个求和
print('x.max():', x.max())
print('np.nan == np.nan ?', np.nan == np.nan)   # False!
print('np.isnan(np.nan) ?', np.isnan(np.nan))   # True

x: [ 1. nan  3.]
x + 10: [11. nan 13.]
x.sum(): nan
x.max(): nan
np.nan == np.nan ? False
np.isnan(np.nan) ? True


### nan-safe 函数家族

为了避免“一个 NaN 毁所有”，NumPy 提供了 **nan-safe 版本**：遇到 NaN 自动跳过。常见的有一整套：

| 普通版 | nan-safe 版 | 说明 |
|--------|-------------|------|
| `np.sum` | `np.nansum` | 求和时忽略 NaN |
| `np.mean` | `np.nanmean` | 求均值时忽略 NaN |
| `np.std` / `np.var` | `np.nanstd` / `np.nanvar` | 标准差 / 方差 |
| `np.min` / `np.max` | `np.nanmin` / `np.nanmax` | 最值 |
| `np.median` | `np.nanmedian` | 中位数 |
| `np.prod` | `np.nanprod` | 连乘 |
| `np.percentile` | `np.nanpercentile` | 分位数 |

> 💡 注意：没有“保留 NaN”的累积版本，累积函数遇到 NaN 时会一路带到后面。一般先 `np.where(np.isnan(x), 0, x)` 再累积。

In [11]:
x = np.array([1.0, np.nan, 3.0, np.nan, 5.0])

print('sum 普通:', x.sum())            # NaN
print('nansum:', np.nansum(x))        # 1+3+5
print('mean 普通:', x.mean())         # NaN
print('nanmean:', np.nanmean(x))      # (1+3+5)/3
print('nanstd:', np.nanstd(x))
print('nanmax:', np.nanmax(x))

sum 普通: nan
nansum: 9.0
mean 普通: nan
nanmean: 3.0
nanstd: 1.632993161855452
nanmax: 5.0


### 检测与填充

处理 NaN 的标准三步：
1. **检测**：`np.isnan(x)` 得到布尔掩码；
2. **定位**：`np.where(np.isnan(x))` 拿到缺失的位置（坐标）；
3. **填充**：用 `np.where` 或布尔索引，把 NaN 替换成 0、均值或插值。

> ⚠️ 填充均值时注意：均值本身也要用 nan-safe 版本算（`nanmean`），否则会被 NaN 污染。

In [12]:
x = np.array([1.0, np.nan, 3.0, np.nan, 5.0])

mask = np.isnan(x)
print('缺失掩码:', mask)
print('缺失个数:', mask.sum())
print('缺失坐标:', np.where(mask))

# 方法一：np.where 填 0
x_fill0 = np.where(mask, 0, x)
print('填 0:', x_fill0)

# 方法二：用均值填充（均值用 nanmean 计算）
x_fill_mean = np.where(mask, np.nanmean(x), x)
print('填均值:', x_fill_mean, ' 均值=', np.nanmean(x))
print('填充后是否还有 NaN:', np.isnan(x_fill_mean).any())

缺失掩码: [False  True False  True False]
缺失个数: 2
缺失坐标: (array([1, 3]),)
填 0: [1. 0. 3. 0. 5.]
填均值: [1. 3. 3. 3. 5.]  均值= 3.0
填充后是否还有 NaN: False


### np.inf 与 np.nan 的对比

`inf`（无穷大）和 `nan`（非数值）是两个“特殊数字”，容易混淆：

| 维度 | `np.inf` | `np.nan` |
|------|----------|----------|
| 含义 | 无穷大（超出浮点范围 / 除零） | 缺失、非法、无法表示 |
| 参与运算 | 会扩散（`inf + 1 = inf`） | 会传染（结果都是 nan） |
| 与自身比较 | `inf == inf` 为 True | `nan == nan` 为 False |
| 检测 | `np.isinf` | `np.isnan` |
| 大小比较 | 可以比较大小 | 与任何数比大小都是 False |

In [13]:
with np.errstate(divide='ignore', invalid='ignore'):
    print('1.0 / 0.0 =', np.float64(1.0) / np.float64(0.0))   # inf
print('inf + 1 =', np.inf + 1)
print('inf - inf =', np.inf - np.inf)     # inf - inf 未定义 -> nan
print('inf * 0 =', np.inf * 0)            # nan
print('inf == inf ?', np.inf == np.inf)
print('nan > 0 ?', np.nan > 0, '  nan < 0 ?', np.nan < 0)    # 都是 False

arr = np.array([1.0, np.inf, np.nan, 3.0])
print()
print('isinf:', np.isinf(arr))
print('isnan:', np.isnan(arr))
print('isfinite:', np.isfinite(arr))   # 只有 1.0 和 3.0 是有限数

1.0 / 0.0 = inf
inf + 1 = inf
inf - inf = nan
inf * 0 = nan
inf == inf ? True
nan > 0 ? False   nan < 0 ? False

isinf: [False  True False False]
isnan: [False False  True False]
isfinite: [ True False False  True]


## 5.5　数学函数补充

第 3 章我们用过 `np.sqrt`、`abs` 等，这里把剩下的常用数学函数补齐。它们全部是**逐元素（element-wise）**的向量化函数。

### ① 三角函数与角度换算

- `sin / cos / tan`：**弧度制**
- `arcsin / arccos / arctan`：反函数
- `degrees` / `radians`：弧度 ↔ 角度 互转

In [14]:
# 注意：NumPy 三角函数默认用“弧度”
rad = np.radians(30)                 # 30 度 -> 弧度
print('30° 的弧度:', rad)
print('sin(30°) =', np.sin(rad))

deg = np.degrees(np.pi / 2)          # π/2 弧度 -> 角度
print('π/2 的度数:', deg)

angles_deg = np.array([0, 30, 45, 60, 90])
print('cos(deg):', np.cos(np.radians(angles_deg)))

30° 的弧度: 0.5235987755982988
sin(30°) = 0.49999999999999994
π/2 的度数: 90.0
cos(deg): [1.00000000e+00 8.66025404e-01 7.07106781e-01 5.00000000e-01
 6.12323400e-17]


### ② 指数与对数

- `exp`：自然指数 eˣ
- `log`：自然对数 ln；`log2`、`log10`：不同底
- `log1p`：计算 log(1+x)，**x 很小时比 log(1+x) 精确得多**

> 💡 **log1p 的意义**：浮点数精度有限，当 x 非常小（如 1e-16）时，`1 + x` 在计算机里可能直接被“四舍五入”成 1，于是 `log(1+x)` 算出 0。`log1p` 内部用更精确的算法，能保留这个小量。金融利率、概率换算中常用。

In [15]:
x = np.array([1, 2, 4, 8, 16])
print('exp:', np.exp(np.array([0.0, 1.0, 2.0])))
print('log:', np.log(x))
print('log2:', np.log2(x))
print('log10:', np.log10(x))

# log1p 的数值精度演示
tiny = 1e-16
print()
print('log(1 + 1e-16) =', np.log(1.0 + tiny))    # 直接被舍入成 0
print('log1p(1e-16)   =', np.log1p(tiny))        # 保留了精确值 ~1e-16

exp: [1.         2.71828183 7.3890561 ]
log: [0.         0.69314718 1.38629436 2.07944154 2.77258872]
log2: [0. 1. 2. 3. 4.]
log10: [0.         0.30103    0.60205999 0.90308999 1.20411998]

log(1 + 1e-16) = 0.0
log1p(1e-16)   = 1e-16


### ③ 其他常用：hypot 等

`hypot(a, b) = sqrt(a² + b²)`，专门用来算直角三角形的斜边长 / 向量长度，既快又稳（避免平方溢出）。

In [16]:
a, b = 3.0, 4.0
print('hypot(3,4):', np.hypot(a, b))            # 5.0
print('sqrt(a^2+b^2):', np.sqrt(a**2 + b**2))   # 等价但略慢且易溢出

# 批量算多个向量的长度
xs = np.array([1.0, 2.0, 3.0])
ys = np.array([4.0, 5.0, 6.0])
print('三个向量的长度:', np.hypot(xs, ys))

hypot(3,4): 5.0
sqrt(a^2+b^2): 5.0
三个向量的长度: [4.12310563 5.38516481 6.70820393]


## 5.6　线性代数入门：`A @ B` vs `A * B`（重点！）

这是本章**最容易翻车**的地方，务必分清：

| 符号 | 含义 | 维度要求 | 结果形状 |
|------|------|----------|----------|
| `A * B` | **逐元素相乘**（对应位置相乘） | 广播规则 | 与 A、B 广播后相同 |
| `A @ B` | **矩阵乘法**（行 × 列 求和） | A 的列数 = B 的行数 | (A行数, B列数) |

> ⚠️ 数学课本里的 “A×B” 通常是矩阵乘；而 NumPy 里的 `*` 是逐元素乘。**写代码前先问自己：我要的是哪种乘？**

> 💡 用 `@` 运算符（Python 3.5+）是官方推荐写法，等价于 `np.matmul(A, B)`。

In [17]:
A = np.array([[1, 2],
              [3, 4]])
B = np.array([[5, 6],
              [7, 8]])

print('A * B（逐元素相乘）:')
print(A * B)
print()
print('A @ B（矩阵乘法）:')
print(A @ B)
print()
print('np.matmul(A, B) 等价:')
print(np.matmul(A, B))

A * B（逐元素相乘）:
[[ 5 12]
 [21 32]]

A @ B（矩阵乘法）:
[[19 22]
 [43 50]]

np.matmul(A, B) 等价:
[[19 22]
 [43 50]]


### np.dot 与 @ 对一维 / 二维的不同行为

- 二维 @ 二维：矩阵乘
- **一维 @ 一维：内积（点积），结果是标量**
- 一维 @ 二维 / 二维 @ 一维：向量-矩阵乘

`np.dot` 的行为与 `@` 类似但历史更久；在“矩阵乘法”场景下官方建议统一用 `@`。

In [18]:
v = np.array([1, 2, 3])
w = np.array([4, 5, 6])

print('一维 @ 一维（内积）:', v @ w)      # 1*4+2*5+3*6 = 32
print('np.dot(v, w):', np.dot(v, w))
print('v * w（逐元素）:', v * w)         # 对比

M = np.array([[1, 0], [0, 1]])
print()
print('二维 @ 一维（矩阵作用到向量）:', M @ v[:2])

一维 @ 一维（内积）: 32
np.dot(v, w): 32
v * w（逐元素）: [ 4 10 18]

二维 @ 一维（矩阵作用到向量）: [1 2]


## 5.7　linalg 核心函数：det / inv / rank / norm / eig / solve

`np.linalg` 是 NumPy 的线性代数子模块，包含矩阵分解、求逆、解方程、特征值等。下面逐个介绍核心成员，并给出一张“遇到什么问题用哪个函数”的速查图：

```mermaid
flowchart TD
    Q{"你要解决什么问题？"} --> Q1["判断矩阵是否可逆 / 面积体积"]
    Q1 --> F1["det 行列式<br/>det ≠ 0 则可逆"]
    Q --> Q2["求逆矩阵"]
    Q2 --> F2["inv"]
    Q --> Q3["矩阵的“秩”多大 / 是否满秩"]
    Q3 --> F3["matrix_rank"]
    Q --> Q4["衡量向量/矩阵大小"]
    Q4 --> F4["norm 范数"]
    Q --> Q5["研究特征值 / 特征向量"]
    Q5 --> F5["eig"]
    Q --> Q6["解线性方程组 A x = b"]
    Q6 --> F6["solve（优先）<br/>而不是 inv(A) @ b"]
    Q --> Q7["数据拟合 / 最小二乘"]
    Q7 --> F7["lstsq"]
```

In [19]:
# det 行列式：二维下 |A| = ad - bc
A = np.array([[3.0, 1.0],
              [2.0, 2.0]])
print('A =')
print(A)
print('det(A):', np.linalg.det(A))          # 3*2 - 1*2 = 4

# 奇异矩阵（行列式为 0）不可逆
S = np.array([[1.0, 2.0],
              [2.0, 4.0]])
print('det(S):', np.linalg.det(S))

A =
[[3. 1.]
 [2. 2.]]
det(A): 4.000000000000001
det(S): 0.0


In [20]:
A = np.array([[3.0, 1.0],
              [2.0, 2.0]])
A_inv = np.linalg.inv(A)
print('A_inv:')
print(A_inv)

# 验证：A @ A_inv 应等于单位阵
I = A @ A_inv
print()
print('A @ A_inv（应为单位阵）:')
print(np.round(I, 10))
print('matrix_rank(A):', np.linalg.matrix_rank(A))
print('matrix_rank(奇异矩阵):', np.linalg.matrix_rank(np.array([[1., 2.], [2., 4.]])))

A_inv:
[[ 0.5  -0.25]
 [-0.5   0.75]]

A @ A_inv（应为单位阵）:
[[1. 0.]
 [0. 1.]]
matrix_rank(A): 2
matrix_rank(奇异矩阵): 1


### norm：向量的“长度”，有不同定义

| 范数 | 公式 | 直觉 | 常见用途 |
|------|------|------|----------|
| L1 范数 | Σ|xᵢ| | “曼哈顿距离” | 稀疏正则化（Lasso） |
| L2 范数 | √(Σxᵢ²) | 欧氏距离（默认） | 最常见的“长度” |
| ∞ 范数 | max|xᵢ| | 最大分量 | 数值误差上界 |

In [21]:
v = np.array([3.0, 4.0])
print('L2 范数（默认）:', np.linalg.norm(v))       # 5
print('L1 范数:', np.linalg.norm(v, ord=1))         # 3+4 = 7
print('无穷范数:', np.linalg.norm(v, ord=np.inf))   # 4

M = np.array([[1.0, 2.0],
              [3.0, 4.0]])
print('矩阵的 Frobenius 范数:', np.linalg.norm(M))  # 所有元素平方和开根

L2 范数（默认）: 5.0
L1 范数: 7.0
无穷范数: 4.0
矩阵的 Frobenius 范数: 5.477225575051661


### eig：特征值与特征向量

对方阵 A，若存在**非零**向量 v 和标量 λ 满足 A v = λ v，则 λ 叫特征值、v 叫特征向量。它描述“矩阵作用在某个方向时，只是拉伸而不改变方向”。我们用 `eig` 求出后再验证 `A @ v ≈ λ * v`。

In [22]:
A = np.array([[4.0, 1.0],
              [1.0, 3.0]])            # 对称矩阵 -> 特征值必为实数
eigvals, eigvecs = np.linalg.eig(A)

print('特征值:', eigvals)
print('特征向量（每列一个）:')
print(eigvecs)

# 验证 A @ v ≈ λ * v
for i in range(2):
    lam = eigvals[i]
    v = eigvecs[:, i]                 # 第 i 列是第 i 个特征向量
    lhs = A @ v
    rhs = lam * v
    print(f'第 {i} 组: A@v = {lhs}, λ*v = {rhs}')
    print(f'  误差 = {np.abs(lhs - rhs).max():.2e}')

特征值: [4.61803399 2.38196601]
特征向量（每列一个）:
[[ 0.85065081 -0.52573111]
 [ 0.52573111  0.85065081]]
第 0 组: A@v = [3.92833435 2.42784414], λ*v = [3.92833435 2.42784414]
  误差 = 0.00e+00
第 1 组: A@v = [-1.25227364  2.02622131], λ*v = [-1.25227364  2.02622131]
  误差 = 2.22e-16


### solve：解线性方程组 A x = b

解方程组 `A x = b` 有两种写法，但强烈推荐 `solve`：

| 写法 | 结果 | 缺点 |
|------|------|------|
| `x = np.linalg.inv(A) @ b` | 数学上等价 | 先求逆再乘，**数值误差大、速度慢** |
| `x = np.linalg.solve(A, b)` | 直接求解 | 更精确、更快，官方推荐 |

> 💡 原因：求逆要额外做大量运算，且逆矩阵对舍入误差更敏感；`solve` 用 LU 分解直接消元，又稳又快。

In [23]:
# 方程组：3x + y = 9, x + 2y = 8
A = np.array([[3.0, 1.0],
              [1.0, 2.0]])
b = np.array([9.0, 8.0])

x_solve = np.linalg.solve(A, b)
print('solve 的解:', x_solve)

x_inv = np.linalg.inv(A) @ b
print('inv(A)@b 的解:', x_inv)

# 回代验证 Ax 应等于 b
print('A @ x_solve:', A @ x_solve)

solve 的解: [2. 3.]
inv(A)@b 的解: [2. 3.]
A @ x_solve: [9. 8.]


## 5.8　实战案例：最小二乘线性拟合

前面学的都串起来，做一件真实的事：**给一组带噪声的点，拟合出背后的直线方程**。

假设真实模型是 y = 1.5x + 0.5（斜率 1.5、截距 0.5），我们观测到的 y 还混入了随机噪声。最小二乘 `lstsq` 会找到让“预测误差平方和最小”的斜率和截距。

> 💡 设计矩阵技巧：要同时拟合斜率和截距，把输入 x 拼成两列 `[x, 1]`，这样解出来的两个系数正好是 [斜率, 截距]。

In [24]:
rng = np.random.default_rng(2024)          # 固定种子，保证结果可复现

# 1) 生成带噪声的数据：y = 1.5x + 0.5 + 噪声
n = 50
x = np.linspace(0, 10, n)
true_a, true_b = 1.5, 0.5
noise = rng.normal(0, 1.0, n)              # 均值0、标准差1的高斯噪声
y = true_a * x + true_b + noise

# 2) 构造设计矩阵 X = [x, 1]
X = np.column_stack([x, np.ones_like(x)])

# 3) 最小二乘拟合
sol, residuals, rank, s = np.linalg.lstsq(X, y, rcond=None)
a_fit, b_fit = sol

print(f'拟合斜率 a = {a_fit:.4f}  (真实值 {true_a})')
print(f'拟合截距 b = {b_fit:.4f}  (真实值 {true_b})')

# 4) 用拟合结果预测并看残差
y_pred = X @ sol
rss = residuals[0] if residuals.size else ((y - y_pred) ** 2).sum()
print('残差平方和:', float(rss))
print('残差平方和(手算):', float(((y - y_pred) ** 2).sum()))

拟合斜率 a = 1.4482  (真实值 1.5)
拟合截距 b = 0.7702  (真实值 0.5)
残差平方和: 46.63925678197037
残差平方和(手算): 46.63925678197038


## 5.9　本章小结

### 一句话记忆表

| 主题 | 一句话记忆 |
|------|-----------|
| axis 聚合 | 聚合 = 沿 axis **消灭该维度**；axis=None 全压成标量 |
| keepdims | 聚合后想继续广播，就加 `keepdims=True` 保住长度为 1 的维 |
| 总体 vs 样本 | `ddof=0` 总体、`ddof=1` 样本，默认是总体！ |
| NaN | 一个 NaN 毁所有 → 用 `nanmean / nansum` 等 nan-safe 家族 |
| 判断 NaN | `nan == nan` 是 False，必须用 `np.isnan` |
| 逐元素 vs 矩阵乘 | `A * B` 逐元素、`A @ B` 矩阵乘，别搞混！ |
| 求逆 vs 求解 | 解方程用 `solve`，别用 `inv(A) @ b` |
| 最小二乘 | `lstsq` 一步搞定线性拟合，设计矩阵 `[x, 1]` 同时拟合斜率和截距 |

### 📝 动手练习（3 题）

1. 造一个 (4, 5) 的随机数组，分别求出：每列的最大值、每行的和、全体中位数。再想想每个结果的形状为什么是这样。
2. 有一个含 3 个 NaN 的数组，用 `nanmean` 求出均值，再用这个均值把所有 NaN 填上；最后确认数组里不再有 NaN。
3. 解方程组 `2x + 3y = 8`、`x - y = -1`，并把解回代验证。

### 本章知识地图

```mermaid
mindmap
  root((第5章 统计聚合与线性代数))
    聚合语义
      axis 压缩维度
      axis=None 全数组
    聚合函数
      sum prod
      mean median average
      std var ddof
      min max argmin argmax
      cumsum cumprod
      percentile quantile
      keepdims
    数据清洗
      NaN 传染性
      nan-safe 家族
      isnan 检测与填充
      inf vs nan
    数学函数
      三角函数
      exp log log1p
      hypot
    线性代数
      @ vs *
      det inv rank norm
      eig 验证
      solve vs inv
      最小二乘 lstsq
```

### 下一站

👉 **下一章：`06_随机数排序与高级技巧.ipynb`** —— 我们将掌握现代随机数 API（`default_rng`）、排序与搜索、文件 IO，并用一张 6 章知识地图收尾整个系列。

> 💡 建议：学完本章后，动手把练习做了再进下一章，聚合 + 线性代数是数据分析的地基。